# Mathematical reasoning: candidates, verification, and search


> The previous lecture let the LLM spend extra thought while generating an answer: chain of thought, self-consistency, and test-time compute. Those methods lengthen reasoning, but they leave a more fundamental issue untouched: when a task demands **absolute correctness**, with no room for ambiguity, the model's reasoning must reach a reliability that a strict judge can accept.
>
> This lecture places that issue in the harshest setting: the International Mathematical Olympiad. We walk three routes — AlphaGeometry solves geometry with a neuro-symbolic split, AlphaProof turns proving into a game that reinforcement learning can search, and Gemini Deep Think writes gold-medal proofs in natural language within the contest time limit. On each route we implement a minimal component by hand: a toy geometry symbolic engine, a line-by-line **verifier**, and a search loop that generates candidates, verifies them, and advances.

Consider a classic fake proof that claims 2 equals 1.

Step 1: set a=b.

Step 2: multiply both sides by a, obtaining a²=ab.

Step 3: subtract b² from both sides, obtaining a²−b²=ab−b².

Step 4: factor the left side, obtaining (a+b)(a−b)=b(a−b).

Step 5: divide both sides by (a−b), obtaining a+b=b; substitute a=b to get 2b=b, hence 2=1.

The error is in step 5. Because step 1 set a=b, a−b=0, so step 5 divides by 0, which is undefined — the proof is void from that line onward. The first four steps each hold; only step 5 is illegal.

A human reader spots this because the rule "0 cannot be a divisor" is already in mind; a machine that rewrites symbols line by line, without an explicit check for that rule, will compute all the way to 2=1. That is the split between a math task and a dialogue task: the former has a unique right or wrong, can be strictly **verified**, and an error can be localized to a concrete line; the latter often treats a fluent but wrong reply as roughly correct.

To stand up to strict scoring, this lecture adds two capabilities that are already accessories of the Agent loop in lecture 1: **verification** and search. By the end we will have implemented three things by hand: a toy symbolic engine that derives geometric facts only by rules, a verifier that checks the legality of each proof line, and a search loop that wires them together. We begin with the scoring rules.

## 1. Mathematics: the touchstone of reasoning

This section uses a classic false proof to make the difference between "a machine checking line by line" and "a human judging by eye" concrete. The proof below claims that 2 equals 1, and it is the classic divide-by-zero trick: set a = b, multiply both sides by a to get a² = ab, subtract b² from both sides to get a² − b² = ab − b², that is (a+b)(a−b) = b(a−b). Divide both sides by (a−b) to get a+b = b, substitute a = b to get 2b = b, hence 2 = 1.

If this appeared on an exam script, a human reader would spot the problem at once: because a = b, a − b = 0, and division by 0 is illegal. The difficulty is that a machine that rewrites symbols line by line, unless it checks the divide-by-zero rule, will also compute all the way through. None of the steps in a natural-language proof has been checked by a machine; the error hides in a single line, and no intermediate result that merely looks reasonable can guarantee the conclusion. That is the split between a math task and a dialogue task: the former needs a line-by-line referee, the latter does not.

Substituting concrete numbers for a and b makes this clearer. Set a=1, b=1. Setting a=b is 1=1, which holds; multiplying both sides by a gives a²=ab, that is 1=1, which holds; subtracting b² gives a²−b²=ab−b², that is 0=0, which holds; after factoring, (a+b)(a−b)=b(a−b) becomes 2×0=1×0, that is 0=0, which still holds. Every step so far is valid. The last step is the issue: both sides are divided by (a−b), and a−b=1−1=0, so both sides are divided by 0, which is undefined. All previous equalities are legal; only this division is illegal, and the entire 2=1 error lives in that one line.

A human reader spots this because human background knowledge includes "0 cannot be a divisor" and checks the divisor without being asked. A machine that rewrites symbols line by line has no such background: it treats "divide both sides by a number" as a purely formal rewrite rule, without looking at the value of the divisor, and therefore treats 0 as a divisor and computes to the end. The only way to make the machine stop is to write "division is allowed only when the divisor is nonzero" as an explicit rule, and to judge every line against the rules. That is what a line-by-line referee means: legality is decided by rules, not by the executor filling in common sense.

This example also explains why scoring is all or nothing: the verifier checks line by line, and a single illegal line makes the whole proof fail. Even if the proof reaches the second-to-last line, an illegal rewrite in the middle scores the same as doing nothing: 0 points. Reward can be paid only at the end of this unbroken chain, which is the most direct source of sparse reward.


In [ ]:
import numpy as np
np.random.seed(42)

# Hand calculation: IMO scoring on a single problem — full marks or zero, with no intermediate band
def imo_score(proved):
    """Score one problem: 7 points if a formal proof is verified, otherwise 0."""
    return 7 if proved else 0

cases = [
    ("Proved and verified", True),
    ("Missing last auxiliary", False),
    ("Right idea, illegal line", False),
    ("Not attempted", False),
]
for name, ok in cases:
    print(f"{name:<28} -> {imo_score(ok)} points")

# Hand calculation: size of the proof search space
b = 20          # mean number of candidate actions per state
d = 50          # typical number of steps in a contest proof
print(f"branching factor b={b}, proof depth d={d}, search-tree leaves about {b ** d:.1e}")
print("Key observation: the reward is only 0 or 7, while the search space is on the order of 10 to a few dozen")


In [ ]:
import matplotlib.pyplot as plt

depths = np.array([5, 10, 20, 30, 50])
leaves = 20.0 ** depths

plt.figure(figsize=(6, 4))
plt.semilogy(depths, leaves, marker="o")
plt.xlabel("proof depth d")
plt.ylabel("search tree leaves (log scale)")
plt.title("Sparse reward meets a huge search space")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Letting a neural network propose geometry steps

This section explains how a machine takes a geometry proof one step at a time. The answer is a division of labor — an engine that only follows rules derives facts step by step, and when it gets stuck a neural network supplies a new object. We first look at how the engine works, then at how the two sides connect.

Start with the most basic operation: deriving new facts from known ones. AlphaGeometry hands this kind of reasoning to an engine called DDAR. It only processes symbols according to rules written in advance; it follows the rules as stated and does not "understand" geometry. A program of this kind is a symbolic engine. DDAR has two parts. The Deductive Database (DD) is a set of deduction rules, such as "the base angles of an isosceles triangle are equal" and "corresponding sides of congruent triangles are equal"; Algebraic Reasoning (AR) solves equations of angles, ratios, and distances. We focus on the DD rules.

Each DD rule is written as "if certain conditions hold, then a conclusion follows." For example the condition `isos(A,B,C)` means triangle ABC is isosceles with vertex A, and the conclusion is that angle ABC equals angle ACB. Rules of this form, a conjunction of conditions implying one conclusion, are each definite and checkable: if the conditions hold, the conclusion must hold. The engine applies these rules repeatedly until no new facts appear.

DDAR has a gap: problems that need an auxiliary construction get stuck completely. An auxiliary construction is a point or line added beyond the problem statement, such as taking the midpoint M of BC, or drawing the perpendicular from A to BC. AlphaGeometry uses a 150M-parameter decoder-only transformer that, when the engine is stuck, proposes which auxiliary point to add next, then hands control back to DDAR.

We put this division of labor into a minimal version. The toy geometry symbolic engine below treats one toy theorem: given that triangle ABC is isosceles (AB = AC, vertex A) and M is the midpoint of base BC, prove that angle AMB equals angle AMC. The rule base is:

- R1 `isos(A,B,C)` → `ang_eq(ABC, ACB)`: the base angles of an isosceles triangle are equal
- R2 `isos(A,B,C)` → `seg_eq(AB, AC)`: the definition of isosceles, the two legs are equal
- R3 `mid(M,B,C)` → `seg_eq(BM, CM)`: the definition of midpoint, BM = CM
- R4 `mid(M,B,C)` ∧ `isos(A,B,C)` → `ang_eq(ABC, ABM)`: M lies on BC, so the two angles are the same angle
- R5 `mid(M,B,C)` ∧ `isos(A,B,C)` → `ang_eq(ACB, ACM)`: likewise, the C-end is collinear so the angles coincide
- R6 `ang_eq(x,y)` ∧ `ang_eq(y,z)` → `ang_eq(x,z)`: transitivity of angle equality
- R7 `ang_eq(x,y)` → `ang_eq(y,x)`: symmetry of angle equality
- R8 SAS: `seg_eq(AB,AC)` ∧ `seg_eq(BM,CM)` ∧ `ang_eq(ABM,ACM)` → `congr(ABM,ACM)`
- R9 `congr(ABM,ACM)` → `ang_eq(AMB,AMC)`: corresponding angles of congruent triangles are equal

We first read the rule base against a concrete triangle. Take A=(0,3), B=(−2,0), C=(2,0). Both legs AB and AC have length √((0−(−2))²+(3−0)²)=√13, so triangle ABC is isosceles with vertex A, written isos(A,B,C). The two base angles are ∠ABC and ∠ACB; the law of cosines gives: vector BA=(2,3), BC=(4,0), cos∠ABC=(2×4+3×0)/(√13×4)=2/√13; vector CA=(−2,3), CB=(−4,0), cos∠ACB=(8+0)/(√13×4)=2/√13. The two angles have the same cosine and both lie between 0 and π, so ∠ABC=∠ACB. That is the geometric fact stated by rule R1: the base angles of an isosceles triangle are equal.

Every rule in the base is written as "a conjunction of premises → one conclusion." A rule of this form is a Horn clause. The left side is a set of facts that must hold together; the right side is one new fact. For example R1 is written `isos(A,B,C) → ang_eq(ABC, ACB)`, read as "if triangle ABC is isosceles with vertex A, then angle ABC equals angle ACB." An angle is named by three points, with the middle point as the vertex: ang_eq(ABC, ACB) means ∠ABC=∠ACB, and in the program it is the 6-tuple `("ang_eq","A","B","C","A","C","B")`. The other predicates work the same way: seg_eq(AB, AC) means segment AB equals AC, mid(M,B,C) means M is the midpoint of segment BC, and congr(ABM, ACM) means triangles ABM and ACM are congruent.

The nine rules, one by one. R1 derives equal base angles from isosceles; R2 derives equal legs from isosceles; R3 derives BM=CM from the midpoint. R4 and R5 say that M lies on BC, so ray BM coincides with ray BC, hence ∠ABC and ∠ABM are the same angle, and ∠ACB and ∠ACM are the same angle. R6 and R7 are transitivity and symmetry of angle equality; R8 is the SAS congruence test, whose premises are two adjacent sides and the included angle pairwise equal; R9 derives corresponding equal angles from congruence. The goal proposition is ∠AMB=∠AMC, written ang_eq(AMB, AMC).

Each rule is definite and checkable: if the premises are in the knowledge base, the conclusion can be added, with no exceptions. That is why a symbolic engine can trust its own output — every deduction step is backed by one such plain rule.

In [ ]:
# Hand calculation: version A with no auxiliary point has only isos(A,B,C) and stops after two steps
base_steps = [
    ("R1_base_angles", ("ang_eq", "A", "B", "C", "A", "C", "B")),
    ("R2_isosceles_def", ("seg_eq", "A", "B", "A", "C")),
]
print("Version A (no auxiliary point) deductions:")
for rule, concl in base_steps:
    print(f"  from isos(A,B,C) derive {concl}   by {rule}")

# Hand calculation: after adding the auxiliary construction mid(M,B,C), the chain can continue
print("\nVersion B (add auxiliary point M and the midpoint relation), full hand chain:")
hand_steps = [
    ("R1_base_angles", ("ang_eq", "A", "B", "C", "A", "C", "B")),
    ("R2_isosceles_def", ("seg_eq", "A", "B", "A", "C")),
    ("R3_midpoint_def", ("seg_eq", "B", "M", "C", "M")),
    ("R4_collinear_left", ("ang_eq", "A", "B", "C", "A", "B", "M")),
    ("R5_collinear_right", ("ang_eq", "A", "C", "B", "A", "C", "M")),
    ("R6_angle_trans", ("ang_eq", "A", "B", "C", "A", "C", "M")),
    ("R7_angle_sym", ("ang_eq", "A", "B", "M", "A", "B", "C")),
    ("R6_angle_trans", ("ang_eq", "A", "B", "M", "A", "C", "M")),
    ("R8_SAS", ("congr", "A", "B", "M", "A", "C", "M")),
    ("R9_congr_corr_angles", ("ang_eq", "A", "M", "B", "A", "M", "C")),
]
for rule, concl in hand_steps:
    print(f"  derive {concl}   by {rule}")
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")
print("goal", goal, "is reachable in version B and unreachable in version A")


A symbolic engine does not stop at the two steps we picked by hand. It automates repeated application of the rules; that process is forward deduction. The knowledge base starts from a set of premise facts and repeatedly scans every rule: whenever all premises of a rule match existing facts in the base, the conclusion is added; if a full scan adds no new fact, it stops. The knowledge base at that point is the closure: every fact the rule base can derive.

Walk version A all the way through, with the single premise isos(A,B,C). Round 1: R1 adds ang_eq(ABC,ACB), R2 adds seg_eq(AB,AC); the base now has only one angle-equality fact, so R6 transitivity has no pair to join; R7 symmetrizes ang_eq(ABC,ACB) into ang_eq(ACB,ABC). Round 2: R6 joins ang_eq(ABC,ACB) with ang_eq(ACB,ABC) end to end, obtaining ang_eq(ABC,ABC). Round 3: R6 joins the other way, obtaining ang_eq(ACB,ACB). Round 4: no rule produces a new fact, the closure is done, 6 facts in total.

Note that R6 contributes only one new fact per round, because the engine takes the first match of each rule in each round. Transitivity and symmetry of angle equality keep producing trivial facts such as ang_eq(ABC,ABC), "an angle equals itself." Forward deduction does not filter facts by value; it prefers over-generation to omission, which is the cost of writing every step out explicitly.

Version A's closure contains no fact that mentions M: the premises have no M, and the rules only combine existing symbols into new facts, so rules that involve M never fire and the goal is unreachable. Version B also puts mid(M,B,C) into the premises; R3 fires BM=CM, and the later chain runs through congruence to the goal — the 10-step hand calculation in the code cell above is the path the engine will take.


In [ ]:
# From scratch: a toy geometry symbolic engine (Horn-clause forward deduction)
def match(pattern, fact, env):
    """Match a pattern against a fact. Variables look like '?X'. Return a binding dict or None."""
    if env is None:
        return None
    if isinstance(pattern, str) and pattern.startswith("?"):
        if pattern in env:
            return env if env[pattern] == fact else None
        env = dict(env)
        env[pattern] = fact
        return env
    if isinstance(pattern, tuple):
        if not isinstance(fact, tuple) or len(pattern) != len(fact):
            return None
        for p, f in zip(pattern, fact):
            env = match(p, f, env)
            if env is None:
                return None
        return env
    return env if pattern == fact else None

def instantiate(pattern, env):
    """Replace variables in a pattern with concrete values from a binding dict."""
    if isinstance(pattern, str):
        return env.get(pattern, pattern)
    if isinstance(pattern, tuple):
        return tuple(instantiate(p, env) for p in pattern)
    return pattern

def enumerate_matches(antecedents, facts, idx, env):
    """Enumerate environments in which all premises hold on the fact base, for forward chaining."""
    if idx == len(antecedents):
        yield env
        return
    for fact in facts:
        new_env = match(antecedents[idx], fact, env)
        if new_env is not None:
            yield from enumerate_matches(antecedents, facts, idx + 1, new_env)

class GeoEngine:
    """Toy geometry symbolic engine: apply Horn rules until no new facts appear."""

    def __init__(self, rules):
        self.rules = rules  # each rule is (name, list of premise patterns, conclusion pattern)

    def forward_chain(self, premises, max_rounds=300):
        """Forward deduction: from the premises derive every derivable fact. Return (facts, trace)."""
        facts = list(premises)
        trace = []
        for _ in range(max_rounds):
            added = False
            for name, antecedents, conclusion in self.rules:
                for env in enumerate_matches(antecedents, facts, 0, {}):
                    new = instantiate(conclusion, env)
                    if new not in facts:
                        facts.append(new)
                        trace.append((name, new))
                        added = True
                        break
            if not added:
                break
        return facts, trace

# Smoke test: pattern matching and instantiation
print("example match:",
      match(("mid", "?M", "?B", "?C"), ("mid", "M", "B", "C"), {}))
print("example instantiate:",
      instantiate(("seg_eq", "?B", "?M"), {"?B": "B", "?M": "M"}))


In [ ]:
RULES = [
    ("R1_base_angles", [("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?B", "?C", "?A", "?C", "?B")),
    ("R2_isosceles_def", [("isos", "?A", "?B", "?C")],
     ("seg_eq", "?A", "?B", "?A", "?C")),
    ("R3_midpoint_def", [("mid", "?M", "?B", "?C")],
     ("seg_eq", "?B", "?M", "?C", "?M")),
    ("R4_collinear_left",
     [("mid", "?M", "?B", "?C"), ("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?B", "?C", "?A", "?B", "?M")),
    ("R5_collinear_right",
     [("mid", "?M", "?B", "?C"), ("isos", "?A", "?B", "?C")],
     ("ang_eq", "?A", "?C", "?B", "?A", "?C", "?M")),
    ("R6_angle_trans",
     [("ang_eq", "?A", "?B", "?C", "?X", "?Y", "?Z"),
      ("ang_eq", "?X", "?Y", "?Z", "?P", "?Q", "?R")],
     ("ang_eq", "?A", "?B", "?C", "?P", "?Q", "?R")),
    ("R7_angle_sym", [("ang_eq", "?A", "?B", "?C", "?X", "?Y", "?Z")],
     ("ang_eq", "?X", "?Y", "?Z", "?A", "?B", "?C")),
    ("R8_SAS",
     [("seg_eq", "?A", "?B", "?A", "?C"),
      ("seg_eq", "?B", "?M", "?C", "?M"),
      ("ang_eq", "?A", "?B", "?M", "?A", "?C", "?M")],
     ("congr", "?A", "?B", "?M", "?A", "?C", "?M")),
    ("R9_congr_corr_angles", [("congr", "?A", "?B", "?M", "?A", "?C", "?M")],
     ("ang_eq", "?A", "?M", "?B", "?A", "?M", "?C")),
]

engine = GeoEngine(RULES)
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")
facts, trace = engine.forward_chain([
    ("isos", "A", "B", "C"),
    ("mid", "M", "B", "C"),
])
print("all facts derived by the engine:")
for f in facts:
    print("  ", f)
print("goal", goal, "was derived:", goal in facts)
print("trace length:", len(trace))


The engine above automates deriving new facts. Now return to auxiliary constructions. Drop mid(M,B,C) from the premises and keep only isos(A,B,C); the engine can derive equal base angles, equal legs, and their symmetric forms, then it stops — M does not exist in the knowledge base, no rule that mentions M can fire, and the goal angles AMB and AMC never come into view. That is the stuck state of DDAR.

AlphaGeometry's response is: when the engine is stuck, let a language model propose an auxiliary construction, add the new fact to the premises, and let the engine continue. This division of labor is the pattern used throughout the lecture — the neural network proposes candidates (auxiliary points, auxiliary lines), and the symbolic engine verifies deterministically and advances. Verification is guaranteed by the engine: the LLM only proposes adding a midpoint M and asserts BM = CM; every later deduction is confirmed one by one by a Horn rule, and the LLM's output never skips a step.

The engine will never think of adding a midpoint, and the reason is in the nature of forward deduction. A rule can fire only if its premises are already facts in the base, and symbols in the base can come only from the original premises or from previously derived conclusions. Taking a midpoint M means introducing into the proof a point that exists in neither the premises nor the conclusions; that is creation, not deduction, and no Horn rule can do it. If M is not in the knowledge base, the premise mid(?M,?B,?C) of R3 never matches.

That is why an auxiliary construction must be proposed by a component outside the engine. A point or line added beyond the problem statement is equivalent to introducing, in search, a new object that did not exist before: first decide what to add, then let the engine reason around it. AlphaGeometry lets a language model make that decision — when the engine is stuck, the model proposes a candidate auxiliary construction, such as taking the midpoint M of BC, and hands mid(M,B,C) to the engine as a new fact; the engine then deducts, and if the goal is reachable the construction is accepted, otherwise the model tries another.

The split is therefore clear: the language model is responsible for invention, proposing objects that are not in the knowledge base; the symbolic engine is responsible for checking, verifying every deduction step. The model can guess wrong and the engine will block it; the engine does not err, but it will also never propose a new point on its own.

In [ ]:
# Experiment: no auxiliary point vs an auxiliary construction, comparing the derived fact sets
goal = ("ang_eq", "A", "M", "B", "A", "M", "C")

facts_a, trace_a = engine.forward_chain([("isos", "A", "B", "C")])
print("Version A (no auxiliary point) derived", len(facts_a), "facts:")
for f in facts_a:
    print("  ", f)
print("goal derived:", goal in facts_a)

facts_b, trace_b = engine.forward_chain([
    ("isos", "A", "B", "C"),
    ("mid", "M", "B", "C"),          # auxiliary construction: midpoint M of base BC
])
print("\nVersion B (add auxiliary point M and the midpoint relation) derived", len(facts_b), "facts:")
for f in facts_b:
    print("  ", f)
print("goal derived:", goal in facts_b)
print("number of new facts:", len(facts_b) - len(facts_a))


In [ ]:
# Visualization: neuro-symbolic split — the language model proposes an auxiliary construction, the symbolic engine deducts deterministically
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")

def box(x, y, w, h, text, color):
    p = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02",
                       facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)

box(0.02, 0.52, 0.30, 0.30, "symbolic engine\nDD: horn rules\n(stuck)", "#dbe9f7")
box(0.38, 0.52, 0.30, 0.30, "LLM\npropose auxiliary\nmid(M, B, C)", "#e7f2d8")
box(0.74, 0.52, 0.24, 0.30, "engine\nresumes\n(deduce)", "#dbe9f7")
box(0.34, 0.06, 0.34, 0.22, "verified proof\nang_eq(AMB, AMC)", "#fbe6d5")

ax.annotate("", xy=(0.38, 0.67), xytext=(0.32, 0.67),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(0.74, 0.67), xytext=(0.68, 0.67),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("", xy=(0.51, 0.28), xytext=(0.86, 0.52),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.text(0.33, 0.72, "propose aux", fontsize=9)
ax.text(0.69, 0.72, "feed facts", fontsize=9)
ax.text(0.72, 0.34, "deduce", fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.title("Neuro-symbolic division of labor in AlphaGeometry")
plt.tight_layout()
plt.show()


## 3. Letting a formal verifier check a proof

AlphaGeometry's symbolic engine is a set of hard-coded domain rules and covers only geometry. This section takes a more general path: first translate the problem and the proof into a format a machine can check line by line, then let a program called a verifier stand guard. We first look at what "line-by-line checking" actually requires.

Natural-language proofs often omit steps with phrases such as "clearly," "it is easy to see," or "by symmetry"; a machine cannot read those omissions, nor decide whether they are justified. To let a machine check line by line, every step of the proof must be written in a strict format, each step corresponding to a rewrite the machine accepts as a rule, and any step without a rule justification is caught on the spot. Writing a proof in that strict format is called formalization; a system that can carry it is a formal proof assistant, and the one AlphaProof uses is Lean.

In a formal system, a proof becomes a game that can be searched. The conclusions reached so far are the position, each action rewrites the position, and the Lean checker is the referee: a legal step is admitted, an illegal step is rejected at once. AlphaZero-style reinforcement learning searches and self-plays in this game; AlphaProof uses it to push IMO problems to silver-medal level.

This path has a bottleneck: human-written formal proofs are too few to train a model. AlphaProof uses a fine-tuned Gemini as a "formalizer," automatically translating natural-language problems into formal statements, building a million-scale bank of problems at varying difficulty, then letting the solver strengthen itself on the hard ones. We cannot reproduce that large-scale training, but we can reproduce the minimal core it depends on: a line-by-line proof verifier, and a search loop that generates candidates, verifies them, and advances.

We first implement the verifier. The toy theory below treats even numbers: the predicate even(n) means there exists an integer k such that n = 2k. The claim to prove is: if even(a) and even(b), then even(a+b). A proof object is a sequence of lines, each line a (statement, rule, dependency line numbers) triple, and the verifier confirms that every line is legal. A hand calculation:

The difference between a formal proof and a natural-language proof is clearest on the 8-step proof above. Line 6, "2k+2l=2(k+l)," is a single sentence in natural language, and a reader accepts it at once by the distributive law; in a formal system, either the distributive law is written as an axiom and cited by a rule, or the machine treats it as algebraic normalization and decides it automatically. The k on line 3 is similar: the definition of even(a) only says some integer k exists, and in natural language "take such a k" rests on that existence guarantee; a formal system must introduce a variable that has not appeared before, and check that it is indeed new.

The cost of making everything explicit is verbosity: a lemma that takes three lines on an exam script often becomes dozens of lines after formalization. AlphaProof pays that cost because it buys a machine that can search the proof space automatically: the proof becomes a game, the Lean checker supplies the rules, and a searcher can try, fail, and backtrack without ever taking an illegal step.

Lines 3 and 4 use witness variables. even(a) means "there exists an integer k such that a=2k"; to unfold the definition into the equation a=2k, a concrete k must first be chosen. In a formal system that k cannot be any name that has already appeared; it must be brand new, otherwise "there exists some number" is miswritten as "it is that number." The verifier's witness rule checks this with is_fresh: the k on line 3 has not appeared in the first two lines, and the l on line 4 has not appeared in the first three, so each passes. Line 8 reclaims the definition using k+l as a new witness, showing that even(a+b) holds.

Line 6 depends on no premise line; it relies on normalization. Its job is to decide whether two algebraic terms are identically equal: reduce 2k+2l and 2(k+l) both to a canonical form, then compare. The method is to expand addition into a list of subterms and sort, and to distribute a constant multiple over addition. 2k+2l lands directly as the two subterms {2k, 2l}; 2(k+l) is first distributed to 2k+2l, then reduced to the same list. The two expressions land on the same canonical form, so line 6 is judged true without citing other lines. That is what the arith rule means: it only checks whether the two sides are equal after normalization.

Normalization is introduced to offload the everyday burden of algebraic identity. Handing "these two expressions are really the same" to a fixed small algorithm is cheaper than writing it into derivation rules one by one, and it does not leave dozens of lines of formalization noise at every such step. The output of normalize in the next code cell will show that these two expressions do land on the same canonical form.


In [ ]:
# Hand calculation: under normalization, 2k + 2l is identical to 2(k + l)
def var(name):
    """Build a variable term."""
    return ("var", name)

def add(t1, t2):
    """Build an addition term."""
    return ("add", t1, t2)

def mul(c, t):
    """Build a constant-multiple term; c is an integer."""
    return ("mul", c, t)

def normalize(t):
    """Reduce a term to canonical form, used to decide algebraic identity."""
    if t[0] == "var":
        return t
    if t[0] == "add":
        parts = []
        for sub in t[1:]:
            n = normalize(sub)
            if n[0] == "add":
                parts.extend(n[1:])
            else:
                parts.append(n)
        return ("add",) + tuple(sorted(parts))
    if t[0] == "mul":
        c, inner = t[1], normalize(t[2])
        if inner[0] == "add":
            subs = [normalize(("mul", c, s)) for s in inner[1:]]
            return normalize(("add",) + tuple(subs))
        if c == 1:
            return inner
        return ("mul", c, inner)
    return t

k, l = var("k"), var("l")
left = add(mul(2, k), mul(2, l))          # 2k + 2l
right = mul(2, add(k, l))                 # 2(k + l)
print("left  =", normalize(left))
print("right =", normalize(right))
print("identical under normalization:", normalize(left) == normalize(right))


In [ ]:
# From scratch: a line-based proof verifier — check rules, dependencies, and normalized equality, line by line
def vars_of(t):
    """Collect every variable name that appears in a term or statement."""
    if isinstance(t, tuple):
        if t[0] == "var":
            return {t[1]}
        out = set()
        for sub in t[1:]:
            out |= vars_of(sub)
        return out
    return set()

def is_fresh(proof, i, premises, name):
    """Whether the witness variable name is completely unused before line i."""
    seen = set()
    for stmt in premises:
        seen |= vars_of(stmt)
    for j in range(i):
        seen |= vars_of(proof[j][1])
    return name not in seen

def check_line(proof, i, premises):
    """Check whether line i is legally derived from its dependencies by a rule. Return (ok, reason)."""
    rule, stmt, deps = proof[i]
    if rule == "premise":
        return stmt in premises, "must be a given premise"
    if rule == "arith":
        return (stmt[0] == "eq" and
                normalize(stmt[1]) == normalize(stmt[2])), "equal after normalization"
    if rule == "witness":
        if len(deps) != 1:
            return False, "witness needs exactly one dependency line"
        dep_stmt = proof[deps[0]][1]
        t = stmt[1]
        if dep_stmt[0] != "even" or normalize(dep_stmt[1]) != normalize(t):
            return False, "dependency must be even(t)"
        if stmt[0] != "eq" or stmt[2][0] != "mul" or stmt[2][1] != 2:
            return False, "statement must be t = 2k"
        w = stmt[2][2]
        if w[0] != "var" or not is_fresh(proof, i, premises, w[1]):
            return False, "witness variable must be fresh"
        return True, "witness unfolding"
    if rule == "compose":
        if len(deps) != 2:
            return False, "addition substitution needs two dependency lines"
        d1, d2 = (proof[j][1] for j in deps)
        if d1[0] != "eq" or d2[0] != "eq":
            return False, "dependencies must be equations"
        if stmt[0] != "eq":
            return False, "statement must be an equation"
        if (normalize(stmt[1]) != normalize(("add", d1[1], d2[1]))
                or normalize(stmt[2]) != normalize(("add", d1[2], d2[2]))):
            return False, "statement does not match the substituted sum"
        return True, "addition substitution"
    if rule == "trans":
        if len(deps) != 2:
            return False, "transitivity needs two dependency lines"
        d1, d2 = (proof[j][1] for j in deps)
        if d1[0] != "eq" or d2[0] != "eq":
            return False, "dependencies must be equations"
        if (normalize(d1[2]) != normalize(d2[1])
                or normalize(stmt[1]) != normalize(d1[1])
                or normalize(stmt[2]) != normalize(d2[2])):
            return False, "middle terms do not connect"
        return True, "equality transitivity"
    if rule == "even_def":
        if len(deps) != 1:
            return False, "reclaiming the definition needs exactly one dependency line"
        dep_stmt = proof[deps[0]][1]
        if (dep_stmt[0] != "eq" or dep_stmt[2][0] != "mul"
                or dep_stmt[2][1] != 2):
            return False, "dependency must be t = 2s"
        if stmt[0] != "even" or normalize(stmt[1]) != normalize(dep_stmt[1]):
            return False, "statement must be even(t)"
        return True, "reclaim definition"
    return False, "unknown rule"

def verify(proof, premises):
    """Check the whole proof line by line. Return (all legal, per-line report)."""
    ok_all = True
    report = []
    for i in range(len(proof)):
        ok, reason = check_line(proof, i, premises)
        report.append((i, proof[i][0], ok, reason))
        if not ok:
            ok_all = False
    return ok_all, report

# Smoke test: commutativity of addition holds automatically under normalization
probe = [("arith", ("eq", add(var("a"), var("b")),
                    add(var("b"), var("a"))), [])]
print("smoke test:", check_line(probe, 0, []))


In [ ]:
# Build a line-based proof of even(a) ∧ even(b) → even(a+b) and run the verifier
a, b = var("a"), var("b")
k, l = var("k"), var("l")
premises = [("even", a), ("even", b)]

proof = [
    ("premise",  ("even", a),                                     []),
    ("premise",  ("even", b),                                     []),
    ("witness",  ("eq", a, ("mul", 2, k)),                        [0]),
    ("witness",  ("eq", b, ("mul", 2, l)),                        [1]),
    ("compose",  ("eq", ("add", a, b),
                  ("add", ("mul", 2, k), ("mul", 2, l))),         [2, 3]),
    ("arith",    ("eq", ("add", ("mul", 2, k), ("mul", 2, l)),
                  ("mul", 2, ("add", k, l))),                     []),
    ("trans",    ("eq", ("add", a, b),
                  ("mul", 2, ("add", k, l))),                     [4, 5]),
    ("even_def", ("even", ("add", a, b)),                         [6]),
]

ok_all, report = verify(proof, premises)
for i, rule, ok, reason in report:
    print(f"line {i}: rule {rule:<8} legal={ok}  ({reason})")
print("whole proof legal:", ok_all)


In [ ]:
# Tampering demo: change the conclusion of line 5 to a + b = 2k; the verifier must reject on the spot
bad = [list(line) for line in proof]
bad[5][1] = ("eq", ("add", a, b), ("mul", 2, k))
ok_bad, report_bad = verify(bad, premises)
print("does the tampered proof pass:", ok_bad)
for i, rule, ok, reason in report_bad:
    if not ok:
        print(f"  illegal line {i}: rule {rule} rejected ({reason})")


The verifier has shown something essential: every step of a proof can be checked by a machine line by line. Now return to AlphaProof. After formalization, a mathematical proof becomes a game that can be searched: the state is the conclusions reached so far, an action is instantiating a rule for the next step, and the verifier is the game rule. AlphaProof searches this game with reinforcement learning; we do the same thing in a more primitive way — implement a generic search loop: from the current state generate candidate steps, let the verifier stand guard, advance on a pass and try another candidate on a failure, until the goal is derived or the space is exhausted.

The abstract framework is demonstrated below on an artificial implication graph. Six propositions p0 through p5, rules r1 through r6 for implications among them, with r3 leading to the dead end p4 and r6 leading to the goal p5. Starting from the initial conclusion p0, every search step is confirmed by the verifier.


Think of a proof as a game. The conclusions reached so far are the position, an action is picking a rule instance for the next step, the rule table is the set of legal actions, and the verifier is the referee: an action whose premise is not the current position, or whose conclusion disagrees with the rule, is judged illegal. The goal position is deriving the claim to be proved. AlphaZero-style reinforcement learning self-plays on this game and trains a policy network from win/loss signals; here we demonstrate the most primitive way to play the same game — depth-first search.

Walk the small implication graph that contains a dead end. The rules are r1: p0→p1, r2: p0→p2, r3: p1→p4, r4: p1→p3, r5: p2→p3, r6: p3→p5. The p4 that r3 leads to has no further rule, so it is a dead end; the goal is p5. Depth-first search keeps pending states on a stack: each step pops the top, finishes if it equals the goal, otherwise pushes all its successors and pops the next. p4 has no successor, so after it is popped the search can only backtrack to p3. The whole process is the table below.

| step | popped state | available actions | pushed | stack | note |
|:---|:---|:---|:---|:---|:---|
| 1 | p0 | r1→p1, r2→p2 | p2, p1 | [p2, p1] | explore the p1 branch first |
| 2 | p1 | r3→p4, r4→p3 | p3, p4 | [p2, p3, p4] | explore p4 first |
| 3 | p4 | none | none | [p2, p3] | dead end, backtrack |
| 4 | p3 | r6→p5 | p5 | [p2, p5] | advance along r6 |
| 5 | p5 | goal | none | [p2] (discarded) | hit the goal, return path p0→p1→p3→p5 |

Dead-end backtracking happens at step 3: p4 has no available action, is pushed and then popped, and does not appear in the final path. Note that p2 was pushed at step 1, but the p1 branch found the goal first, so p2 is never popped — depth-first search stops when it finds one path and does not guarantee visiting every state. The search visits 5 nodes, matching the output of the code cell that follows.

Every search step still goes through the verifier. In pure DFS, candidates are generated from the rule table, so they already satisfy "the premise is the current state and the conclusion matches the rule"; the code still feeds the final path through verify_step line by line, simulating the verifier's role in AlphaProof. Later, when a language model is attached, the model will offer arbitrary candidates, and this verification layer goes from a formality to a necessity — that is where the verifier actually does its work.

Real proof search is far larger than this small graph: the example in section 1 gave a branching factor of about 20 and a depth of about 50, and the full space is astronomical. AlphaZero's idea is to let a neural network rank candidates, steer search toward promising branches, and then rely on the verifier to confirm; the DFS here has no ranking, and is the most primitive instance of the same framework.


In [ ]:
# From scratch: an abstract loop of search plus a verifier
SEARCH_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r3": ("p1", "p4"),   # dead end: p4 has no further rule
    "r4": ("p1", "p3"),
    "r5": ("p2", "p3"),
    "r6": ("p3", "p5"),   # leads to the goal
}

def applicable(state, rules):
    """Generator: return every rule instance applicable from the current conclusion."""
    cands = []
    for name, (premise, conclusion) in rules.items():
        if premise == state:
            cands.append((name, conclusion))
    return cands

def verify_step(state, conclusion, rule, rules):
    """Verifier: legal only if the rule premise is the current state and the conclusion matches the rule."""
    if rule not in rules:
        return False, "unknown rule"
    premise, expected = rules[rule]
    if premise != state:
        return False, "premise is not the current state"
    if conclusion != expected:
        return False, "conclusion does not match the rule"
    return True, ""

def search_dfs(start, goal, rules, max_depth=8):
    """Depth-first search for a proof path, backtracking from dead ends. Return (path, nodes visited)."""
    stack = [(start, [])]
    visited = {start}
    nodes = 0
    while stack:
        state, path = stack.pop()
        nodes += 1
        if state == goal:
            return path, nodes
        if len(path) >= max_depth:
            continue
        for name, conclusion in reversed(applicable(state, rules)):
            if conclusion not in visited:
                visited.add(conclusion)
                stack.append((conclusion, path + [(name, conclusion)]))
    return None, nodes

def check_path(path, rules, start, goal):
    """Feed a proof path to the verifier line by line. Return (all legal, reached goal)."""
    state = start
    for name, conclusion in path:
        ok, _ = verify_step(state, conclusion, name, rules)
        if not ok:
            return False, state == goal
        state = conclusion
    return True, state == goal

# Smoke test: which candidates the generator can offer from the current state
print("applicable candidates from p0:", applicable("p0", SEARCH_RULES))


In [ ]:
# Run search: from p0 find a proof path to p5
path, nodes = search_dfs("p0", "p5", SEARCH_RULES)
print("nodes visited by search:", nodes)
for step, (name, conclusion) in enumerate(path):
    print(f"  step {step + 1}: apply {name} -> derive {conclusion}")

legal, reached = check_path(path, SEARCH_RULES, "p0", "p5")
print("path verified line by line:", legal)
print("goal reached:", reached)
print("dead end p4 was visited then backtracked, and is absent from the final path:",
      "p4" not in [c for _, c in path])


In [ ]:
# Visualization: search on the implication graph; green is the found path, red is the dead end
pos = {
    "p0": (0.0, 0.0),
    "p1": (-1.0, -1.0),
    "p2": (1.0, -1.0),
    "p4": (-1.9, -2.0),
    "p3": (0.0, -2.0),
    "p5": (0.5, -3.0),
}
edges = [
    ("r1", "p0", "p1", "path"), ("r2", "p0", "p2", "gray"),
    ("r3", "p1", "p4", "dead"), ("r4", "p1", "p3", "path"),
    ("r5", "p2", "p3", "gray"), ("r6", "p3", "p5", "path"),
]

fig, ax = plt.subplots(figsize=(6, 4.5))
for name, u, v, kind in edges:
    (x1, y1), (x2, y2) = pos[u], pos[v]
    color = {"gray": "#b0b0b0", "dead": "#d64541", "path": "#2e8b57"}[kind]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.8))
    ax.text((x1 + x2) / 2 + 0.08, (y1 + y2) / 2, name, fontsize=9, color=color)
for node, (x, y) in pos.items():
    is_dead = node == "p4"
    ax.plot(x, y, "o", markersize=11, color="#d64541" if is_dead else "#4c9be8")
    ax.text(x + 0.1, y + 0.06, node, fontsize=10)
ax.text(-1.0, -2.7, "p4: dead end (backtrack)", fontsize=8, color="#d64541")
ax.text(0.05, -3.5, "p5: goal reached", fontsize=8, color="#2e8b57")
ax.set_xlim(-2.5, 2.2)
ax.set_ylim(-3.7, 0.4)
ax.axis("off")
plt.title("DFS on the implication graph (green = found path)")
plt.tight_layout()
plt.show()


## 4. Trade-offs between natural-language reasoning and formal proof

This section steps back and compares two technical routes: one hands reliable verification to a machine, the other drops formalization and leaves right or wrong to contest judges. We first look at the scores of each, then at the division of labor both routes still share.

AlphaProof's route puts reliable verification first: the problem is translated into Lean, every proof step is confirmed by a machine, and the cost is formal translation plus days of compute. Gemini Deep Think takes the other path — it drops formalization entirely, reads the official statement, writes a rigorous proof in natural language, and finishes within the contest time limit. At IMO 2025 it reached the gold line with 35/42 (5 of 6 problems correct), officially certified by IMO coordinators against the same standard used for human contestants. That year it overtook its predecessor: in 2024 AlphaProof and AlphaGeometry 2 together scored 28/42, and still needed two or three days of compute.

The trade-off between the two routes is worth stating clearly. The formal-verification route aims at absolute reliability: the verifier lets no illegal step through, but it is slow and expensive, and the problem must first be formalized. The natural-language RL route aims at speed and generality: no translation is needed, the model reads the problem directly, but verifying whether a proof is correct remains an open problem, and the final score is given by people. In AlphaProof's proof search the model proposes candidate steps and a formal verifier stands guard; that neuro-symbolic split is kept on both routes — the difference is only whether the guard is a person or a machine.

Below we attach llm_client to the search framework of the previous section: the model proposes the next proof step, the formal verifier checks each one, and the loop runs until the proof is done or fails. Under the scripted demo the model's output is a placeholder, and the demo code runs entirely offline.

In [ ]:
# Unified LLM client: use a real model when an API key is present, otherwise fall back to a scripted demo
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()

# Convention for model output: STEP: <conclusion> RULE: <rule name>; the verifier checks line by line against this
import re

def parse_steps(text):
    """Parse a list of (conclusion, rule name) from model output, format STEP: X RULE: r."""
    return re.findall(r"STEP:\s*(\S+)\s*RULE:\s*(\S+)", text)

# Under the scripted demo, a scripted trace exercises parsing and the verification path (a live API would emit full reasoning)
scripted = (
    "STEP: p1 RULE: r1\n"
    "STEP: p2 RULE: r5\n"      # illegal: r5 needs premise p2, current state is p1
    "STEP: p3 RULE: r4\n"
    "STEP: p5 RULE: r6\n"
)
steps = parse_steps(scripted)
print("scripted demo output is a placeholder, parsed step count:", len(steps))

state = "p0"
report = []
for idx, (conclusion, rule) in enumerate(steps):
    ok, reason = verify_step(state, conclusion, rule, SEARCH_RULES)
    report.append((idx, conclusion, rule, ok, reason))
    if ok:
        state = conclusion
for idx, conclusion, rule, ok, reason in report:
    print(f"step {idx}: {conclusion} / {rule}  legal={ok}  ({reason})")
failed = [idx for idx, _, _, ok, _ in report if not ok]
print("indices of steps judged illegal:", failed)


In [ ]:
def propose_next(state, goal, rules, client):
    """Let the model propose the next proof step. Under the scripted demo, one-step lookahead avoids dead ends."""
    if False:
        for name, (premise, conclusion) in rules.items():
            if premise != state:
                continue
            if conclusion == goal or applicable(conclusion, rules):
                return [(conclusion, name)]
        for name, (premise, conclusion) in rules.items():
            if premise == state:
                return [(conclusion, name)]
        return []
    prompt = (
        "Current state: " + state + "\n"
        "Goal: " + goal + "\n"
        "Rule base: " + ", ".join(
            f"{n}: {p} -> {c}" for n, (p, c) in rules.items()) + "\n"
        "Output the next step, format STEP: <conclusion> RULE: <rule name>"
    )
    reply = client.chat([{"role": "user", "content": prompt}], temperature=0.2)
    return parse_steps(reply)

def run_proof_loop(start, goal, rules, client, max_rounds=10):
    """The LLM proposes the next step, the verifier stands guard, loop until the proof is done or rounds run out."""
    state = start
    log = []
    for rnd in range(max_rounds):
        if state == goal:
            return log, True
        for conclusion, rule in propose_next(state, goal, rules, client):
            ok, reason = verify_step(state, conclusion, rule, rules)
            if ok:
                state = conclusion
                log.append((rnd, conclusion, rule, "accepted"))
                break
            log.append((rnd, conclusion, rule, "rejected: " + reason))
    return log, state == goal

log, done = run_proof_loop("p0", "p5", SEARCH_RULES, client)
for rnd, conclusion, rule, status in log:
    print(f"round {rnd}: candidate {conclusion} / {rule} -> {status}")
print("proof complete:", done)


In [ ]:
# Visualization: comparing the two routes — formal verification vs natural-language RL
routes = ["formal Lean\nIMO 2024\n(~2-3 days)",
          "natural language\nIMO 2025\n(within 4.5 h)"]
scores = [28, 35]

fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(routes, scores, color=["#9bb7d4", "#7ed6a5"])
ax.axhline(29, color="#d64541", linestyle="--", linewidth=1.2)
ax.text(0.9, 30, "gold threshold 29/42", color="#d64541", fontsize=9)
for b, s in zip(bars, scores):
    ax.text(b.get_x() + b.get_width() / 2, s + 0.5, str(s),
            ha="center", fontsize=11)
ax.set_ylim(0, 42)
ax.set_ylabel("score out of 42")
ax.set_title("Two routes to IMO gold")
plt.tight_layout()
plt.show()


**Where the two routes meet**

Putting the three papers together yields a clear evolution: a pure symbolic engine can only solve a fixed domain; adding a neural proposer brings the ability to make auxiliary constructions (AlphaGeometry); replacing the symbolic engine with a formal proof system and pairing it with reinforcement-learning search covers arbitrary mathematical propositions (AlphaProof); when the model and the reinforcement learning are strong enough, the formalization layer can even be dropped, the model reads the problem and writes a natural-language proof, and takes gold within the time limit (Gemini). For an Agent course, the point of mathematics is that it gives any Agent that wants to get stronger at reasoning a verifiable field for training and evaluation. The next lecture assembles these components into an autonomous agent, and reliability returns as a central issue.

## Summary

- [ ] Mathematics is the touchstone of reasoning: verifiable, sparse-reward, and in need of search and tools; together the three leave no room for vague argument
- [ ] Toy geometry symbolic engine: Horn-clause rules forward-deduce until closure; DD and AR are the two halves of DDAR
- [ ] Auxiliary constructions unlock derivation: without an auxiliary point the engine is stuck; after adding midpoint M the SAS congruence chain reaches the goal
- [ ] Neuro-symbolic split: the language model proposes candidate steps, the symbolic engine verifies deterministically and advances
- [ ] Minimal proof verifier: a line-based proof is checked for rules, dependencies, and normalized equality, and illegal steps are marked on the spot
- [ ] Search + verifier framework: generate candidates, verify, backtrack from dead ends — the skeleton of AlphaProof search
- [ ] Two routes compared: formal verification is reliable but slow, natural-language RL is fast and general; in 2025 Gemini took official gold within the time limit

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

The three short problems all build on components written in this lecture. Fill them in by hand, then run the assertions to check.


**Exercise 1: the verifier checks line by line**

Complete check_line so that it judges whether line i of a proof is legally derived from its dependency lines by a rule. The proof uses the line format (statement, rule name, dependency line numbers); the rule base RULES gives the premises and conclusion of each rule.

Hint: first take the rule name of line i, then check that its premise list equals the statements of the dependency lines and that the conclusion equals this line's statement; the premise rule is admitted directly.


In [ ]:
# Exercise 1: complete check_line so that a legal proof all passes and a tampered line is judged illegal
HW1_RULES = {
    "r1": (["p0"], "p1"),
    "r2": (["p0"], "p2"),
    "r3": (["p1"], "p4"),
    "r4": (["p1"], "p3"),
    "r6": (["p3"], "p5"),
}

hw1_proof = [
    ("p0", "premise", []),
    ("p1", "r1", [0]),
    ("p3", "r4", [1]),
    ("p5", "r6", [2]),
]

def check_line(proof, i, rules):
    """Judge whether line i is legally derived from its dependencies by a rule. Return True/False."""
    stmt, rule, deps = proof[i]
    if rule == "premise":
        return True
    premises, conclusion = rules[rule]
    # fill in: the dependency statements equal the rule premises, and this line's statement equals the rule conclusion
    dep_stmts = [proof[j][0] for j in deps]
    return dep_stmts == premises and stmt == conclusion

assert all(check_line(hw1_proof, i, HW1_RULES) for i in range(len(hw1_proof)))
bad = list(hw1_proof)
bad[2] = ("p3", "r4", [0])      # change the dependency from line 1 to line 0
assert not check_line(bad, 2, HW1_RULES)
print("Takeaway: the verifier checks line by line; a line whose dependencies do not match the rule premises is judged illegal")


**Exercise 2: search for a proof path**

Complete find_proof to run depth-first search on the implication graph, returning a proof path from the initial conclusion to the goal together with the number of nodes visited. The rule base contains a dead-end r3; the search must backtrack to reach the goal.

Hint: maintain the current state and a visited set; at each step collect every rule instance whose premise is the current state and whose conclusion has not been visited; prune when max_depth is exceeded.


In [ ]:
# Exercise 2: complete find_proof, using DFS to find a legal proof path
HW2_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r3": ("p1", "p4"),
    "r4": ("p1", "p3"),
    "r5": ("p2", "p3"),
    "r6": ("p3", "p5"),
}

def find_proof(start, goal, rules, max_depth=8):
    """Depth-first search for a proof path. Return (path, nodes visited); path is None if none is found."""
    stack = [(start, [])]
    visited = {start}
    nodes = 0
    while stack:
        state, path = stack.pop()
        nodes += 1
        if state == goal:
            return path, nodes
        if len(path) >= max_depth:
            continue
        candidates = []
        for name, (premise, conclusion) in rules.items():
            # fill in: select only when the premise is the current state and the conclusion has not been visited
            if premise == state and conclusion not in visited:
                candidates.append((name, conclusion))
        for name, conclusion in reversed(candidates):
            visited.add(conclusion)
            stack.append((conclusion, path + [(name, conclusion)]))
    return None, nodes

path, nodes = find_proof("p0", "p5", HW2_RULES)
assert path is not None
state = "p0"
for name, conclusion in path:
    premise, expected = HW2_RULES[name]
    assert premise == state
    assert conclusion == expected
    state = conclusion
assert state == "p5"
print("Takeaway: DFS found a", len(path), "-step proof, visited", nodes, "nodes, path is", path)


**Exercise 3: parse model output and let the verifier stand guard**

Complete parse_steps so that it parses a list of (conclusion, rule name) from text marked with STEP/RULE, then checks each one with verify_line. The text mixes in one illegal step, and the verifier must mark it.

Hint: use re.findall to match the pattern STEP: <conclusion> RULE: <rule name>, with non-space characters \S+; then verify in order, and update the current state only on a pass.


In [ ]:
# Exercise 3: complete parse_steps, parse model output and verify each step
import re

HW3_RULES = {
    "r1": ("p0", "p1"),
    "r2": ("p0", "p2"),
    "r4": ("p1", "p3"),
    "r6": ("p3", "p5"),
}

hw3_text = (
    "STEP: p1 RULE: r1\n"
    "STEP: p3 RULE: r4\n"
    "STEP: p5 RULE: r6\n"
    "STEP: p2 RULE: r2\n"      # illegal: r2 needs premise p0, current state is p5
)

def parse_steps(text):
    """Parse a list of (conclusion, rule name) from STEP/RULE markers."""
    # fill in: match marker pairs in order of appearance
    return re.findall(r"STEP:\s*(\S+)\s*RULE:\s*(\S+)", text)

def verify_line(state, conclusion, rule, rules):
    """Verify one step: the rule exists and its premise is the current state. Return (ok, reason)."""
    if rule not in rules:
        return False, "unknown rule"
    premise, expected = rules[rule]
    if premise != state:
        return False, "premise is not the current state"
    if conclusion != expected:
        return False, "conclusion does not match the rule"
    return True, ""

steps = parse_steps(hw3_text)
state = "p0"
report = []
for idx, (conclusion, rule) in enumerate(steps):
    ok, reason = verify_line(state, conclusion, rule, HW3_RULES)
    report.append((idx, conclusion, rule, ok, reason))
    if ok:
        state = conclusion

assert len(steps) == 4
failed = [idx for idx, _, _, ok, _ in report if not ok]
assert failed == [3]
for idx, conclusion, rule, ok, reason in report:
    print(f"step {idx}: {conclusion} / {rule}  legal={ok}  ({reason})")
print("Takeaway: every parsed step is fed to the verifier, and illegal steps are marked", failed)


## References

- Trinh et al., [AlphaGeometry: An Olympiad-level AI system for geometry](https://www.nature.com/articles/s41586-023-06747-5), Nature 2024, DOI:10.1038/s41586-023-06747-5 — a neuro-symbolic geometry proving system: a language model proposes auxiliary constructions, the symbolic engine DDAR deducts deterministically, 25/30 on IMO-AG-30
- [AI achieves silver-medal standard solving International Mathematical Olympiad problems](https://deepmind.google/discover/blog/ai-solves-imo-problems-at-silver-medal-level/), DeepMind blog 2024-07-25 — official announcement that AlphaProof and AlphaGeometry 2 scored 28/42 silver at IMO 2024
- [AlphaProof technical report](https://www.nature.com/articles/s41586-025-09833-y), Nature 2025, DOI:10.1038/s41586-025-09833-y — full technical details of the proof network, AND-OR tree search, auto-formalization pipeline, and TTRL
- [Advanced version of Gemini with Deep Think officially achieves gold-medal standard at the IMO](https://deepmind.google/blog/advanced-version-of-gemini-with-deep-think-officially-achieves-gold-medal-standard-at-the-international-mathematical-olympiad/), DeepMind blog 2025-07 — Gemini Deep Think scored 35/42 official gold at IMO 2025
- [The Lean Theorem Prover](https://lean-lang.org/) — the formal proof assistant and dependent-type language that AlphaProof depends on
- [miniF2F](https://github.com/openai/miniF2F) — a 488-problem contest-level formal/informal paired benchmark for evaluating theorem provers and auto-formalizers
